In [4]:
!pip uninstall -y androguard
!pip install androguard==3.3.5


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 922.4/922.4 kB 34.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.0/105.0 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 56.9 MB/s eta 0:00:00


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
import os
import csv
import traceback
import pandas as pd
from tqdm.notebook import tqdm
from androguard.misc import AnalyzeAPK


In [6]:
df = pd.read_csv("/content/drive/MyDrive/00 IITR INTERNSHIP/00_final_dataset.csv")


In [ ]:
BASE_PATH = "/content/drive/MyDrive/00 IITR INTERNSHIP"

In [ ]:
def extract_features(apk_path):

    try:
        apk, dex_files, analysis = AnalyzeAPK(apk_path)

        # ---------------- PERMISSIONS ----------------
        permissions = list(set(
            [p.split('.')[-1] for p in apk.get_permissions()]
        ))
        permission_count = len(permissions)

        # ---------------- API CALLS ----------------
        api_calls = set()
        for dex in dex_files:
            for method in dex.get_methods():
                api_calls.add(method.get_name())

        api_count = len(api_calls)

        # ---------------- INTENTS ----------------
        intents = []
        for receiver in apk.get_receivers():
            filters = apk.get_intent_filters("receiver", receiver)
            if filters:
                for action in filters.get("action", []):
                    intents.append(action)


        intent_count = len(intents)

        # ---------------- OPCODES ----------------
        opcodes = []
        for dex in dex_files:
            for method in dex.get_methods():
                code = method.get_code()
                if code:
                    for ins in code.get_bc().get_instructions():
                        opcodes.append(ins.get_name())

        opcode_count = len(opcodes)

        # Limit opcode length (important for memory + transformer safety)
        opcodes_limited = opcodes[:3000]

        # ---------------- METADATA ----------------
        apk_size = os.path.getsize(apk_path)

        return (
            " ".join(permissions),
            " ".join(api_calls),
            " ".join(intents),
            " ".join(opcodes_limited),
            apk_size,
            permission_count,
            api_count,
            opcode_count,
            intent_count
        )

    except Exception as e:
        return None, None, None, None, None, None, None, None, None


In [ ]:
def process_folder(folder_name):

    folder_path = os.path.join(BASE_PATH, folder_name)
    output_file = os.path.join(BASE_PATH + "/00 Datasets", f"{folder_name.lower()}_dataset2.csv")

    processed_files = set()

    if os.path.exists(output_file):
        df_existing = pd.read_csv(output_file)
        processed_files = set(df_existing["file_name"].tolist())
    else:
        with open(output_file, "w", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)
            writer.writerow([
                "file_name",
                "family",
                "apk_size",
                "permission_count",
                "api_count",
                "opcode_count",
                "intent_count",
                "PERM",
                "API",
                "INTENT",
                "OPCODE",
            ])

    for file in tqdm(os.listdir(folder_path)):

        if not file.endswith(".apk"):
            continue

        if file in processed_files:
            continue

        apk_path = os.path.join(folder_path, file)

        (
            perm_text,
            api_text,
            intent_text,
            opcode_text,
            apk_size,
            perm_count,
            api_count,
            opcode_count,
            intent_count
        ) = extract_features(apk_path)


        if perm_text is not None:
            with open(output_file, "a", newline="", encoding="utf-8") as f:
                writer = csv.writer(f)
                writer.writerow([
                    file,
                    folder_name,
                    apk_size,
                    perm_count,
                    api_count,
                    opcode_count,
                    intent_count,
                    perm_text,
                    api_text,
                    intent_text,
                    opcode_text
                ])


In [ ]:
process_folder("Adware")

  0%|          | 0/236 [00:00<?, ?it/s]

In [ ]:
process_folder("Banking")

In [ ]:
process_folder("Riskware")

In [ ]:
process_folder("SMS")

In [ ]:
process_folder("Benign3")

In [ ]:
import pandas as pd
import glob
import os

# 1️⃣ Folder path
folder_path = "/content/drive/MyDrive/00 IITR INTERNSHIP/00 Datasets"

# 2️⃣ Get all CSV file paths
csv_files = glob.glob(os.path.join(folder_path, "*.csv"))

print("Files found:", csv_files)

# 3️⃣ Read all files into list
dataframes = []

for file in csv_files:
    df = pd.read_csv(file)
    print(f"Loaded {file} with shape {df.shape}")
    dataframes.append(df)

# 4️⃣ Merge all datasets (vertical merge)
merged_df = pd.concat(dataframes, ignore_index=True)

# 5️⃣ Remove duplicates (optional but recommended)
# merged_df = merged_df.drop_duplicates()

print("Final merged shape:", merged_df.shape)

# 6️⃣ Save merged dataset
merged_df.to_csv("final_dataset.csv", index=False)

print("Merged dataset saved successfully.")

Files found: ['/content/drive/MyDrive/00 IITR INTERNSHIP/00 Datasets/banking_dataset.csv', '/content/drive/MyDrive/00 IITR INTERNSHIP/00 Datasets/riskware_dataset.csv', '/content/drive/MyDrive/00 IITR INTERNSHIP/00 Datasets/sms_dataset.csv', '/content/drive/MyDrive/00 IITR INTERNSHIP/00 Datasets/adware_dataset2.csv', '/content/drive/MyDrive/00 IITR INTERNSHIP/00 Datasets/banking_dataset2.csv', '/content/drive/MyDrive/00 IITR INTERNSHIP/00 Datasets/benign3_dataset2.csv', '/content/drive/MyDrive/00 IITR INTERNSHIP/00 Datasets/adware_dataset.csv']
Loaded /content/drive/MyDrive/00 IITR INTERNSHIP/00 Datasets/banking_dataset.csv with shape (805, 11)
Loaded /content/drive/MyDrive/00 IITR INTERNSHIP/00 Datasets/riskware_dataset.csv with shape (892, 11)
Loaded /content/drive/MyDrive/00 IITR INTERNSHIP/00 Datasets/sms_dataset.csv with shape (1089, 11)
Loaded /content/drive/MyDrive/00 IITR INTERNSHIP/00 Datasets/adware_dataset2.csv with shape (235, 11)
Loaded /content/drive/MyDrive/00 IITR INTER

In [ ]:
import os
print(os.getcwd())

/content


In [ ]:
import pandas as pd

file_path = "/content/drive/MyDrive/00 IITR Internship/Datasets/sms_dataset.csv"

df = pd.read_csv(file_path)

# Drop rows where ANY numeric column is 0
numeric_cols = [
    "apk_size",
    "permission_count",
    "api_count",
    "opcode_count",
    "intent_count"
]

df_clean = df[(df[numeric_cols] != 0).all(axis=1)]

df_clean.to_csv(file_path, index=False)

print("Rows with any numeric value = 0 removed.")

Rows with any numeric value = 0 removed.
